# Módulo 09 · Aula 03 — Pipelines e Monitoramento

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Quem faz o deploy é você. Se você estiver de férias, ninguém sobe versão. E ontem alguém subiu sem rodar os testes — de novo."*

E a pior de todas:

> *"A gente descobre que o site caiu quando um cliente liga."*

Duas dores, dois assuntos:

| Dor | Resposta |
|-----|----------|
| Deploy depende de uma pessoa | **Pipeline** — o `git push` faz tudo |
| O cliente avisa antes do sistema | **Monitoramento** — o sistema avisa antes |

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | CI vs CD | Duas coisas com nomes parecidos |
| 2 | **GitHub Actions** | 🎯 Com um executor local de verdade |
| 3 | Portões de qualidade | O que barra um merge |
| 4 | 🔴 Segredos no CI | E o vazamento por log |
| 5 | Deploy automático | E quando NÃO automatizar |
| 6 | Logs estruturados | Achar a agulha |
| 7 | Métricas e os 4 sinais | O que olhar |
| 8 | 🔴 Alertas | E a fadiga que os mata |
| 9 | Checklist de produção | O fechamento |

> 🎯 **Você vai escrever um executor de workflow** que lê o `.github/workflows/ci.yml`, resolve as dependências entre jobs e **executa os passos de verdade** na sua máquina. Assim você testa o pipeline antes de dar `push` — e entende o que o GitHub faz.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 09
# ═══════════════════════════════════════════════════════════════
import json
import os
import platform
import re
import shutil
import socket
import subprocess
import sys
import textwrap
import threading
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"),
               ("uvicorn[standard]", "uvicorn"), ("pyyaml", "yaml"),
               ("gunicorn", "gunicorn")]:
    _garantir(_p, _m)

import httpx
import uvicorn
import yaml
from fastapi import FastAPI

TEM_GUNICORN = _garantir("gunicorn", "gunicorn")
E_LINUX = platform.system() == "Linux"

print(f"sistema   : {platform.system()}")
print(f"gunicorn  : {'✅ disponível' if TEM_GUNICORN else '⚠️ indisponível (só Unix)'}")


# ═══════════════════════════════════════════════════════════════
#  Servidores de verdade
# ═══════════════════════════════════════════════════════════════

def porta_livre() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


class Servico:
    """Sobe uma app ASGI num uvicorn de verdade, em segundo plano."""

    def __init__(self, app, nome: str = "servico", **config):
        self.app, self.nome = app, nome
        self.porta = porta_livre()
        self.url = f"http://127.0.0.1:{self.porta}"
        self.config = config
        self._servidor = self._thread = None

    def iniciar(self, timeout: float = 20.0) -> "Servico":
        cfg = uvicorn.Config(self.app, host="127.0.0.1", port=self.porta,
                             log_level="critical", access_log=False, **self.config)
        self._servidor = uvicorn.Server(cfg)
        self._thread = threading.Thread(target=self._servidor.run, daemon=True)
        self._thread.start()
        limite = time.monotonic() + timeout
        while time.monotonic() < limite:
            if self._servidor.started:
                return self
            time.sleep(0.05)
        raise RuntimeError(f"{self.nome} não subiu")

    def parar(self) -> None:
        if self._servidor is not None:
            self._servidor.should_exit = True
            self._thread.join(timeout=10)

    def __enter__(self):
        return self.iniciar()

    def __exit__(self, *_):
        self.parar()


# ═══════════════════════════════════════════════════════════════
#  Shell e exibição
# ═══════════════════════════════════════════════════════════════

def sh(comando: str, mostrar: bool = True, cwd=None, timeout: int = 120,
       env: dict | None = None) -> subprocess.CompletedProcess:
    p = subprocess.run(comando, shell=True, capture_output=True, text=True,
                       cwd=cwd, timeout=timeout,
                       env=dict(os.environ, **(env or {})))
    if mostrar:
        saida = (p.stdout + p.stderr).rstrip()
        if saida:
            print(saida)
    return p


def referencia(comando: str, esperado: str = "") -> None:
    """Mostra um comando que NÃO roda aqui, com a saída típica.

    🔴 Saída marcada `[referência]` não foi executada. Rode você mesmo —
       é assim que se aprende deploy.
    """
    print(f"$ {comando}")
    if esperado:
        for linha in textwrap.dedent(esperado).strip("\n").splitlines():
            print(f"  {linha}")
    print("  ── [referência] não executado neste ambiente ──")


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def arvore(raiz: Path, prefixo: str = "") -> None:
    itens = [i for i in sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
             if i.name not in {"__pycache__", ".git"}]
    for i, item in enumerate(itens):
        ultimo = i == len(itens) - 1
        print(f"{prefixo}{'└── ' if ultimo else '├── '}{item.name}")
        if item.is_dir():
            arvore(item, prefixo + ("    " if ultimo else "│   "))


print("✅ `Servico`, `sh()`, `referencia()`, `tabela()`, `preparar()` prontos")

## 1. CI e CD

In [ ]:
BASE = preparar("aula_09_03")

print("""
   CI — Integração Contínua
   ════════════════════════
   A cada push: instala, lint, tipos, TESTES, auditoria.
   Objetivo: descobrir que quebrou em MINUTOS, não em dias.

   CD — Entrega/Implantação Contínua
   ═════════════════════════════════
   Passou no CI → publica sozinho.

   ⚠️ "Delivery" e "Deployment" não são a mesma coisa:
        Delivery   → fica PRONTO para publicar; alguém aperta o botão
        Deployment → publica sozinho, sem botão

   💭 A Aurora deveria começar por Delivery. Deployment automático em
      produção exige confiança na suíte de testes — e essa confiança
      se constrói vendo o CI pegar erros de verdade por alguns meses.
""")

etapas = [
    ["1. Lint",        "ruff",                    "~5s",   "estilo e erro provável"],
    ["2. Tipos",       "mypy",                    "~20s",  "erro que o teste não pega"],
    ["3. Testes",      "pytest",                  "~60s",  "🔴 o portão principal"],
    ["4. Cobertura",   "pytest-cov",              "~0s",   "detector de buraco"],
    ["5. Auditoria",   "auditar_containers.py",   "~2s",   "M08"],
    ["6. Segredos",    "gitleaks",                "~10s",  "🔴 credencial no commit"],
    ["7. Build",       "docker build",            "~90s",  "o artefato"],
    ["8. Deploy",      "ssh + symlink",           "~30s",  "M09_02"],
]
tabela(["ETAPA", "FERRAMENTA", "TEMPO", "O QUE PEGA"], etapas, [16, 26, 8, 30])

print("""
🎯 A ORDEM É POR CUSTO CRESCENTE.

   Lint leva 5 segundos e pega erro de digitação. Rodá-lo primeiro faz
   o desenvolvedor receber a resposta em 20 segundos em vez de 4
   minutos.

   💭 É a mesma lógica de validar na fronteira (M06): falhe o mais
      cedo e o mais barato possível.
""")

## 2. 🎯 GitHub Actions — e um executor local

In [ ]:
FLUXOS = BASE / ".github" / "workflows"
FLUXOS.mkdir(parents=True)

CI = FLUXOS / "ci.yml"
CI.write_text("""# ═══════════════════════════════════════════════════════════════
#  CI do Atlas — roda a cada push e a cada pull request
# ═══════════════════════════════════════════════════════════════
name: CI

on:
  push:
    branches: [main]
  pull_request:

env:
  ATLAS_AMBIENTE: teste
  PYTHONDONTWRITEBYTECODE: "1"

jobs:

  # ── Rápido: 5 segundos, pega o erro bobo ──
  qualidade:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - name: Sintaxe de todos os .py
        run: python -m compileall -q src/
      - name: Nenhum print esquecido no código de produção
        run: |
          ! grep -rn "^\\s*print(" src/ --include="*.py" \\
            | grep -v "cli.py" | grep -v "apresentacao.py"

  # ── 🔒 Segurança: antes dos testes, porque é rápido e crítico ──
  seguranca:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Nenhum segredo literal no código
        run: |
          ! grep -rnE '(SECRET|PASSWORD|SENHA|TOKEN|API_KEY)\\w*\\s*=\\s*["'"'"'][^"'"'"']{8,}' \\
            src/ --include="*.py"
      - name: .env não está versionado
        run: '! git ls-files --error-unmatch .env 2>/dev/null'

  # ── O portão principal ──
  testes:
    needs: [qualidade, seguranca]
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Rodar a suíte
        run: python -m pytest -q --tb=short -m "not integracao"

  # ── Só publica se TUDO passou ──
  publicar:
    needs: [testes]
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Publicar
        run: echo "deploy da versão $GITHUB_SHA"
        env:
          CHAVE_DEPLOY: ${{ secrets.CHAVE_DEPLOY }}
""", encoding="utf-8")

print(f"✅ ci.yml ({len(CI.read_text(encoding='utf-8').splitlines())} linhas)")
doc = yaml.safe_load(CI.read_text(encoding="utf-8"))
print(f"   jobs: {list(doc['jobs'])}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Um executor de workflow — roda os passos DE VERDADE
# ═══════════════════════════════════════════════════════════════
def ordenar_jobs(jobs: dict) -> list[str]:
    """Ordenação topológica sobre `needs`.

    🔑 É isto que o GitHub faz por baixo: monta um grafo de
       dependências e executa em ondas. Jobs sem dependência entre si
       rodam em PARALELO lá — aqui rodamos em sequência, por
       simplicidade.
    """
    pendentes, ordem = dict(jobs), []
    while pendentes:
        prontos = [
            nome for nome, job in pendentes.items()
            if all(d in ordem for d in (
                [job["needs"]] if isinstance(job.get("needs"), str)
                else (job.get("needs") or [])))
        ]
        if not prontos:
            raise ValueError(f"🔴 ciclo ou dependência inexistente: {list(pendentes)}")
        for nome in sorted(prontos):
            ordem.append(nome)
            pendentes.pop(nome)
    return ordem


def rodar_workflow(caminho: Path, cwd: Path, segredos: dict | None = None) -> int:
    """Executa um workflow do GitHub Actions localmente.

    ⚠️ O QUE ELE NÃO FAZ (e é honesto dizer):
       · `uses:` — ações do marketplace são apenas anunciadas
       · matriz de versões, cache, artefatos
       · execução em paralelo
       · os filtros de `on:` e `if:`

       Ele cobre o que importa para VOCÊ testar antes do push: a ordem
       dos jobs e os comandos `run:`.
    """
    wf = yaml.safe_load(caminho.read_text(encoding="utf-8"))
    jobs = wf.get("jobs") or {}
    ordem = ordenar_jobs(jobs)

    print(f"workflow: {wf.get('name', '(sem nome)')}")
    print(f"ordem   : {' → '.join(ordem)}\n")

    ambiente = dict(os.environ, **{k: str(v) for k, v in (wf.get("env") or {}).items()})
    # 🔴 Segredos entram como variável de ambiente — igual ao GitHub.
    for chave, valor in (segredos or {}).items():
        ambiente[chave] = valor

    falharam: list[str] = []
    for nome in ordem:
        job = jobs[nome]
        deps = ([job["needs"]] if isinstance(job.get("needs"), str)
                else (job.get("needs") or []))
        if any(d in falharam for d in deps):
            print(f"⏭️  {nome}: PULADO (dependência falhou)\n")
            falharam.append(nome)
            continue

        print(f"┌─ {nome}")
        env_job = dict(ambiente, **{k: str(v) for k, v in (job.get("env") or {}).items()})
        tudo_ok = True

        for passo in job.get("steps") or []:
            if "uses" in passo:
                print(f"│  ⏭️  uses: {passo['uses']}")
                continue
            comando = passo.get("run")
            if not comando:
                continue
            rotulo = passo.get("name") or comando.splitlines()[0][:44]
            env_passo = dict(env_job,
                             **{k: str(v) for k, v in (passo.get("env") or {}).items()})
            inicio = time.perf_counter()
            r = subprocess.run(comando, shell=True, cwd=cwd, capture_output=True,
                               text=True, env=env_passo)
            ms = (time.perf_counter() - inicio) * 1000
            print(f"│  {'✅' if r.returncode == 0 else '🔴'} {rotulo:<46}{ms:>7.0f}ms")
            if r.returncode != 0:
                for linha in (r.stdout + r.stderr).strip().splitlines()[-5:]:
                    print(f"│       {linha}")
                if not passo.get("continue-on-error"):
                    tudo_ok = False
                    break
        print(f"└─ {'✅' if tudo_ok else '🔴'}\n")
        if not tudo_ok:
            falharam.append(nome)

    return 1 if falharam else 0


print("✅ `rodar_workflow()` pronta")

In [ ]:
# Um projeto de mentira, mas com código de verdade
PROJETO = BASE / "projeto"
(PROJETO / "src" / "atlas").mkdir(parents=True)
(PROJETO / "tests").mkdir()

(PROJETO / "src" / "atlas" / "__init__.py").write_text("", encoding="utf-8")
(PROJETO / "src" / "atlas" / "regras.py").write_text('''
"""Regras de preço da Aurora."""
import os

CHAVE = os.environ.get("ATLAS_SECRET_KEY", "")


def margem(preco: float, custo: float) -> float:
    if preco <= 0:
        raise ValueError("preço deve ser positivo")
    return round((preco - custo) / preco * 100, 1)
''', encoding="utf-8")

(PROJETO / "tests" / "test_regras.py").write_text('''
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent.parent / "src"))

import pytest
from atlas.regras import margem


def test_margem_normal():
    assert margem(100.0, 60.0) == 40.0


def test_margem_zero():
    assert margem(100.0, 100.0) == 0.0


@pytest.mark.parametrize("preco", [0, -1])
def test_preco_invalido(preco):
    with pytest.raises(ValueError, match="positivo"):
        margem(preco, 10.0)
''', encoding="utf-8")

sh(f'cd "{PROJETO}" && git init -q && git add -A && '
   f'git -c user.email=ci@atlas -c user.name=CI commit -qm inicial',
   mostrar=False)

shutil.copytree(BASE / ".github", PROJETO / ".github")
print("✅ projeto pronto\n")
arvore(PROJETO)

In [ ]:
print("═══ RODANDO O PIPELINE DE VERDADE ═══\n")
codigo = rodar_workflow(PROJETO / ".github" / "workflows" / "ci.yml",
                        cwd=PROJETO,
                        segredos={"CHAVE_DEPLOY": "***valor-secreto***"})
print(f"[código de saída: {codigo}]  →  "
      f"{'✅ CI verde' if codigo == 0 else '🔴 CI vermelho'}")

> 🎯 **Repare no que acabou de acontecer:** o `pytest` rodou de verdade, o `grep` de segredos rodou de verdade, e a ordem `qualidade, seguranca → testes → publicar` foi resolvida a partir do `needs:`.
>
> 💡 **Poder rodar o pipeline antes do push muda o ciclo.** Sem isso, cada ajuste no `ci.yml` custa um commit, um push e três minutos de espera — e você acaba com um histórico cheio de "fix ci", "fix ci 2", "agora vai".

In [ ]:
# 🔴 E quando alguém quebra alguma coisa?
print("═══ Alguém commitou um segredo ═══\n")

(PROJETO / "src" / "atlas" / "config_ruim.py").write_text(
    'ATLAS_SECRET_KEY = "chave-de-producao-vazada-123456"\n', encoding="utf-8")

codigo = rodar_workflow(PROJETO / ".github" / "workflows" / "ci.yml", cwd=PROJETO)
print(f"[código de saída: {codigo}]  →  "
      f"{'✅ CI verde' if codigo == 0 else '🔴 CI vermelho — merge barrado'}")

(PROJETO / "src" / "atlas" / "config_ruim.py").unlink()

> 🎯 **Repare no `⏭️ PULADO`.** Como `testes` depende de `seguranca`, e `seguranca` falhou, o resto nem roda.
>
> Isso não é só economia de tempo: é o pipeline dizendo **qual** é o problema. Se todos os jobs rodassem e falhassem, você teria três telas vermelhas e nenhuma pista de por onde começar.

In [ ]:
# 🔴 E se um teste quebrar?
print("═══ Alguém quebrou uma regra de negócio ═══\n")

regras = PROJETO / "src" / "atlas" / "regras.py"
original = regras.read_text(encoding="utf-8")
marca_tempo = regras.stat().st_mtime          # guardamos a data original

# Troca a divisão: a margem passa a ser calculada sobre o custo.
# ⚠️ Repare que `preco` e `custo` têm o MESMO número de letras.
regras.write_text(original.replace("(preco - custo) / preco", "(preco - custo) / custo"),
                  encoding="utf-8")

# ⚠️ E devolvemos a data de modificação original.
#
#    Isto NÃO é artifício para o exemplo dar certo — é a reprodução
#    determinística de uma situação que acontece sozinha quando a
#    escrita cai no mesmo segundo da compilação anterior. Sem forçar,
#    o resultado seria intermitente: às vezes verde, às vezes vermelho.
#
#    💭 E "às vezes" é exatamente o que torna esse bug tão difícil de
#       diagnosticar na vida real.
os.utime(regras, (marca_tempo, marca_tempo))

codigo = rodar_workflow(PROJETO / ".github" / "workflows" / "ci.yml", cwd=PROJETO)
print(f"[código de saída: {codigo}]  →  "
      f"{'✅ CI verde' if codigo == 0 else '🔴 CI vermelho'}")

> 🔴 **O CI ficou VERDE — e o código está errado.**
>
> `margem(100, 60)` agora devolve `66.7` em vez de `40.0`. O teste existe, cobre exatamente esse caso, e mesmo assim passou.
>
> Isto não é um exemplo inventado: **aconteceu comigo ao escrever esta aula**, e o diagnóstico levou um bom tempo.
>
> 💭 **Antes de ler a resposta, pense:** o que pode fazer o Python executar um código diferente do que está no arquivo?

In [ ]:
# ═══ O diagnóstico ═══
print("O arquivo REALMENTE mudou?\n")
linha = [l for l in regras.read_text(encoding="utf-8").splitlines()
         if "return round" in l][0]
print(f"   {linha.strip()}")
print("   ✅ mudou. Então o problema não é a escrita.\n")

cache = PROJETO / "src" / "atlas" / "__pycache__"
print("O que existe em __pycache__:")
for f in sorted(cache.glob("*.pyc")):
    print(f"   {f.name}  ({f.stat().st_size} bytes)")

print(f"""
🔴 AQUI ESTÁ.

   O job `qualidade` roda `python -m compileall -q src/`, que GRAVA
   bytecode em __pycache__.

   O Python decide se um .pyc está velho comparando DOIS campos do
   arquivo-fonte: a data de modificação e o TAMANHO.

   E a nossa alteração trocou `preco` por `custo` — cinco letras por
   cinco letras. O arquivo tem EXATAMENTE o mesmo tamanho:
""")
print(f"   antes : {len(original)} bytes")
print(f"   depois: {len(regras.read_text(encoding='utf-8'))} bytes")
print("""
   E a data de modificação também bate. Para o Python, o .pyc continua
   válido — e ele executa o bytecode ANTIGO.

   🎯 O teste passou porque testou o código velho.
""")

print("Confirmando pelos metadados que o Python compara:\n")
pyc = next(cache.glob("regras.*.pyc"))
print(f"   mtime do .py  : {regras.stat().st_mtime:.0f}")
print(f"   tamanho do .py: {regras.stat().st_size} bytes")
print(f"   o .pyc guarda esses dois valores no cabeçalho e os compara.")
print(f"   Ambos batem → 'está atualizado' → usa o bytecode velho.")

In [ ]:
# ═══ A correção ═══
print("Removendo o bytecode e rodando de novo:\n")
shutil.rmtree(cache, ignore_errors=True)

codigo = rodar_workflow(PROJETO / ".github" / "workflows" / "ci.yml", cwd=PROJETO)
print(f"[código de saída: {codigo}]  →  "
      f"{'✅ CI verde' if codigo == 0 else '🔴 CI vermelho — como devia'}")

regras.write_text(original, encoding="utf-8")
shutil.rmtree(cache, ignore_errors=True)

> 🎯 **A lição vale muito mais que o bug.**
>
> **1. É por isso que o CI começa de um clone limpo.** O `actions/checkout@v4` baixa o repositório do zero, sem `__pycache__`, sem `.venv`, sem artefato de execução anterior. Um runner que reaproveita o diretório pode testar código que não existe mais.
>
> **2. É por isso que `PYTHONDONTWRITEBYTECODE=1` está no `env:` do workflow** — e por que ele está também no `Dockerfile` do M08.
>
> **3. O que mais me chamou a atenção:** o teste não deu erro. Não houve exceção, nem aviso, nem nada no log. Ele simplesmente **verificou a coisa errada e disse que estava tudo bem**.
>
> 🔴 **É a mesma família de problema do M07:** uma verificação que nunca reprova não está verificando nada. A diferença é que aqui ela reprovaria — só que estava olhando para o passado.
>
> 💡 **Se um teste passar quando você tem certeza de que deveria falhar**, suspeite do cache antes de suspeitar de você. `find . -name __pycache__ -exec rm -rf {} +` é barato.

## 3. 🔴 Segredos no CI

In [ ]:
print("""
   COMO O GITHUB GUARDA

     Settings → Secrets and variables → Actions

     · criptografados em repouso
     · não aparecem em pull request de fork
     · MASCARADOS no log: viram ***

   COMO USAR

     env:
       CHAVE_DEPLOY: ${{ secrets.CHAVE_DEPLOY }}
""")

print("""🔴 O MASCARAMENTO NÃO É MÁGICA — ELE COMPARA TEXTO

   O GitHub substitui ocorrências EXATAS do valor por `***`. Se o
   segredo passar por qualquer transformação, ele escapa:

     echo $CHAVE | base64          🔴 vaza (o valor mudou)
     echo $CHAVE | rev             🔴 vaza
     curl -H "Auth: $CHAVE" -v     🔴 o -v imprime o cabeçalho
     set -x                        🔴 imprime TODOS os comandos
     env                           🔴 lista todas as variáveis

   ⚠️ E log de CI costuma ser público em repositório aberto.
""")

# Demonstração do que o mascaramento pega e não pega
import base64

SEGREDO = "chave-super-secreta-123456"


def mascarar(texto: str, segredo: str) -> str:
    """O que o GitHub faz: substitui o valor exato."""
    return texto.replace(segredo, "***")


casos = [
    ("echo $CHAVE",              SEGREDO),
    ("echo $CHAVE | base64",     base64.b64encode(SEGREDO.encode()).decode()),
    ("echo ${CHAVE:0:8}",        SEGREDO[:8]),
    ("echo $CHAVE | rev",        SEGREDO[::-1]),
]
print("O que apareceria no log:\n")
for comando, saida in casos:
    mascarado = mascarar(saida, SEGREDO)
    vazou = SEGREDO not in mascarado and mascarado != "***"
    marca = "🔴 VAZOU" if vazou else "✅ mascarado"
    print(f"   {comando:<26} → {mascarado[:34]:<36} {marca}")

> 🔴 **`echo ${CHAVE:0:8}` vaza oito caracteres — e o GitHub não percebe**, porque a substring não é o valor exato.
>
> 🧭 **As regras:**
>
> 1. Nunca imprima um segredo, nem "só um pedaço para conferir"
> 2. `set -x` nunca num job que usa segredo
> 3. `curl -v` também não
> 4. Segredo só existe no job que precisa dele
> 5. 🔴 **Se vazou, ROTACIONE.** Apagar o log não basta — ele já foi indexado, copiado, e está no cache de alguém.

In [ ]:
print("""
🔒 OIDC — O JEITO MODERNO, SEM SEGREDO NENHUM

   Em vez de guardar uma chave de longa duração, o GitHub prova a
   identidade do workflow para o provedor de nuvem, que devolve uma
   credencial temporária (minutos de validade).

     permissions:
       id-token: write
     steps:
       - uses: aws-actions/configure-aws-credentials@v4
         with:
           role-to-assume: arn:aws:iam::123:role/atlas-deploy

   💭 O segredo que não existe não pode vazar. Onde o seu provedor
      suportar OIDC, prefira-o.
""")

## 4. Deploy automático — e quando não

In [ ]:
print("""
   ✅ AUTOMATIZE quando:
        · a suíte de testes é confiável (M07)
        · o rollback é de um comando (09_02)
        · há homologação antes
        · o deploy é reversível

   🔴 NÃO AUTOMATIZE (ainda) quando:
        · os testes são poucos ou lentos demais para rodar sempre
        · não existe rollback
        · a migração é destrutiva
        · é sexta-feira às 18h

💭 "Não faça deploy na sexta" não é superstição — é gestão de risco.
   O custo de um problema não é só o conserto; é quem está disponível
   para consertar. Se a sua equipe é você, sexta à noite é o pior
   momento possível.

   🎯 E a resposta boa não é "nunca faça deploy na sexta". É tornar o
      deploy tão seguro que o dia da semana deixe de importar. Isso se
      chama confiança, e ela se constrói com testes e rollback.
""")

CD = FLUXOS / "cd.yml"
CD.write_text("""name: CD

on:
  workflow_run:
    workflows: [CI]
    types: [completed]
    branches: [main]

jobs:
  publicar:
    # 🔑 Só se o CI passou. Sem isto, um CI vermelho publicaria.
    if: github.event.workflow_run.conclusion == 'success'
    runs-on: ubuntu-latest

    # 🔒 `environment` habilita APROVAÇÃO MANUAL e restringe segredos.
    #    É assim que se faz "Continuous Delivery": pronto para
    #    publicar, mas alguém aperta o botão.
    environment:
      name: producao
      url: https://atlas.aurora.com.br

    steps:
      - uses: actions/checkout@v4

      - name: Configurar a chave SSH
        run: |
          mkdir -p ~/.ssh
          echo "${{ secrets.CHAVE_DEPLOY }}" > ~/.ssh/id_ed25519
          chmod 600 ~/.ssh/id_ed25519
          ssh-keyscan -H ${{ secrets.SERVIDOR }} >> ~/.ssh/known_hosts
          # ⚠️ `ssh-keyscan` aceita a chave do servidor sem verificar.
          #    Num ambiente sério, fixe o fingerprint esperado.

      - name: Deploy
        run: ./deploy.sh producao
        env:
          SERVIDOR: ${{ secrets.SERVIDOR }}

      - name: Verificar a saúde
        run: |
          for i in $(seq 1 20); do
            curl -fsS --max-time 3 https://atlas.aurora.com.br/saude && exit 0
            sleep 3
          done
          exit 1

      # 🔴 Se a verificação falhou, reverte SOZINHO.
      - name: Rollback automático
        if: failure()
        run: ssh deploy@${{ secrets.SERVIDOR }} /opt/atlas/rollback.sh
""", encoding="utf-8")

print(f"✅ cd.yml ({len(CD.read_text(encoding='utf-8').splitlines())} linhas)")
print("\n🔑 As três linhas que importam:")
print("   · `if: ...conclusion == 'success'`  → só publica se o CI passou")
print("   · `environment: producao`           → aprovação manual")
print("   · `if: failure()` no rollback       → reverte sozinho")

## 5. Logs estruturados

In [ ]:
print("""
   TEXTO SOLTO                     ESTRUTURADO (JSON)
   ═══════════                     ══════════════════
   Erro ao processar pedido 9042   {"nivel":"error","evento":"pedido_falhou",
                                    "pedido_id":9042,"erro":"estoque",
                                    "req_id":"a1b2c3","ms":142}

   grep + torcida                  consulta de verdade
""")

print("""💭 A diferença aparece na PERGUNTA que você consegue fazer.

   Com texto solto:  "quantos pedidos falharam por estoque ontem?"
                     → grep + contagem manual + torcer pelo formato

   Com JSON:         evento="pedido_falhou" AND erro="estoque"
                     → resposta em segundos, com gráfico

   🔑 O `observabilidade.py` do M04 já produz JSON. O que falta é
      mandá-lo para o stdout (fator 11) e ter um id de correlação
      (M06) em toda linha.
""")

In [ ]:
import json
import logging
import sys
from datetime import datetime, timezone


class FormatadorJSON(logging.Formatter):
    """Uma linha JSON por evento — pronto para qualquer coletor."""

    def format(self, registro: logging.LogRecord) -> str:
        dados = {
            "hora": datetime.fromtimestamp(registro.created, timezone.utc)
                    .isoformat(timespec="milliseconds"),
            "nivel": registro.levelname.lower(),
            "evento": registro.getMessage(),
            "logger": registro.name,
        }
        # 🔑 Campos extras viram colunas consultáveis
        for chave, valor in getattr(registro, "extra_campos", {}).items():
            dados[chave] = valor
        if registro.exc_info:
            dados["excecao"] = self.formatException(registro.exc_info).splitlines()[-1]
        return json.dumps(dados, ensure_ascii=False)


def registrar(logger, nivel: str, evento: str, **campos):
    getattr(logger, nivel)(evento, extra={"extra_campos": campos})


manipulador = logging.StreamHandler(sys.stdout)
manipulador.setFormatter(FormatadorJSON())
log = logging.getLogger("atlas")
log.handlers.clear()
log.addHandler(manipulador)
log.setLevel(logging.INFO)
log.propagate = False

registrar(log, "info", "requisicao", rota="/produtos", metodo="GET",
          status=200, ms=42, req_id="a1b2c3", usuario="ana@aurora.com.br")
registrar(log, "warning", "estoque_baixo", sku="PE-RED-K552", restante=3)
registrar(log, "error", "pedido_falhou", pedido_id=9042, erro="estoque_insuficiente",
          req_id="a1b2c3")

In [ ]:
# 🔴 O que NÃO pode ir para o log
print("""
🔴 NUNCA REGISTRE

   · senha, token, chave — nem "só os 8 primeiros caracteres"
   · número de cartão, CVV
   · CPF, e-mail, telefone   ← 🔴 LGPD: log é tratamento de dado
   · o corpo inteiro de uma requisição de login
   · cookie de sessão

💡 Se precisar correlacionar por usuário, registre o ID INTERNO
   (`usuario_id: 42`), não o e-mail. Ele identifica sem expor.

⚠️ E lembre: log costuma ir para um serviço de terceiro, ficar meses
   guardado, e ser visível para mais gente do que o banco.
""")

SENSIVEIS = {"senha", "password", "token", "secret", "cartao", "cvv", "cpf", "email"}


def limpar_campos(campos: dict) -> dict:
    """Remove o que não pode ser registrado.

    🔑 Lista de bloqueio é frágil (esquece um campo novo). Numa base
       séria, use lista de PERMISSÃO: só registra o que foi aprovado.
    """
    return {k: ("***" if any(s in k.lower() for s in SENSIVEIS) else v)
            for k, v in campos.items()}


sujo = {"usuario_id": 42, "email": "ana@aurora.com.br",
        "senha": "aurora123", "token_sessao": "eyJhbGc...", "rota": "/auth/token"}
print("antes :", sujo)
print("depois:", limpar_campos(sujo))

## 6. Métricas — os quatro sinais dourados

In [ ]:
sinais = [
    ["Latência",    "quanto demora",        "p95 e p99, nunca a média"],
    ["Tráfego",     "quantas requisições",  "req/s por rota"],
    ["Erros",       "quantas falham",       "% de 5xx"],
    ["Saturação",   "quão cheio está",      "CPU, memória, pool do banco"],
]
tabela(["SINAL", "O QUE É", "COMO MEDIR"], sinais, [14, 26, 34])

print("""
🔴 POR QUE A MÉDIA MENTE

   Se 95 requisições levam 50ms e 5 levam 4 segundos, a média dá
   ~250ms — um número que não descreve NINGUÉM. Nem quem foi bem
   atendido, nem quem esperou 4 segundos.

   O p95 diz: "95% das pessoas esperaram no máximo X". É a experiência
   real de quase todo mundo, e ele denuncia a cauda que a média esconde.
""")

# Provando com números
import statistics

amostra = [50] * 95 + [4000] * 5
amostra.sort()
print(f"   amostra : 95 requisições de 50ms + 5 de 4000ms")
print(f"   média   : {statistics.mean(amostra):>7.0f} ms  ← não descreve ninguém")
print(f"   mediana : {statistics.median(amostra):>7.0f} ms")
print(f"   p95     : {amostra[int(0.95 * len(amostra)) - 1]:>7.0f} ms")
print(f"   p99     : {amostra[int(0.99 * len(amostra)) - 1]:>7.0f} ms  ← 🔴 a dor real")

In [ ]:
print("""
💡 E O SINAL QUE FALTA NA LISTA: MÉTRICA DE NEGÓCIO.

   Os quatro sinais dizem que o SISTEMA está bem. Não dizem que a
   EMPRESA está bem.

   Para a Aurora:
     · pedidos por hora          🔴 caiu a zero? algo quebrou
     · taxa de conversão
     · fila de webhooks pendentes (M07)
     · tempo até o pedido liberar após o pagamento

   💭 Um deploy pode manter latência, erro e CPU perfeitos e mesmo
      assim zerar as vendas — um botão que sumiu, um formulário que
      não envia. Só a métrica de negócio pega isso.

   🎯 Se você só puder monitorar UMA coisa, monitore "pedidos por
      hora". Ela cobre indiretamente quase todo o resto.
""")

## 7. 🔴 Alertas e a fadiga

In [ ]:
print("""
   🔴 O MAIOR PROBLEMA DE MONITORAMENTO NÃO É FALTA DE ALERTA.
      É ALERTA DEMAIS.

   Uma equipe que recebe 40 alertas por dia para de ler os alertas.
   Aí o alerta importante chega e ninguém abre. O sistema de
   monitoramento existe, funciona, e não serve para nada.
""")

alertas = [
    ["% de 5xx > 1% por 5 min",       "🔴 acorda",  "usuário sofrendo agora"],
    ["p95 > 2s por 10 min",           "🔴 acorda",  "degradação real"],
    ["/saude falhando",               "🔴 acorda",  "está fora"],
    ["pedidos/hora = 0 em horário útil", "🔴 acorda", "🎯 negócio parado"],
    ["disco > 85%",                   "⚠️ horário comercial", "dá para planejar"],
    ["certificado expira em 15 dias", "⚠️ horário comercial", "dá tempo"],
    ["CPU > 80% por 5 min",           "📊 só gráfico", "pode ser normal"],
    ["um erro 500 isolado",           "📊 só gráfico", "🔴 NÃO acorda ninguém"],
]
tabela(["CONDIÇÃO", "AÇÃO", "POR QUÊ"], alertas, [36, 22, 26])

print("""
🧭 A REGRA: SÓ ACORDA ALGUÉM O QUE EXIGE AÇÃO IMEDIATA DE UMA PESSOA.

   Antes de criar um alerta, responda:
     1. Se disparar às 3h, alguém consegue FAZER alguma coisa?
     2. Se a resposta for "olhar e voltar a dormir" → não é alerta
     3. Se dispara toda semana sem incidente → o limiar está errado

   💡 Alerta que dispara e não exige ação deve ser APAGADO, não
      ignorado. Um alerta ignorado ensina a ignorar todos.
""")

In [ ]:
# Uma rota de métricas simples
metricas = FastAPI()
CONTADORES = {"requisicoes": 0, "erros": 0, "pedidos": 0}
LATENCIAS: list[float] = []


@metricas.get("/metricas")
def ver_metricas():
    """Formato de texto do Prometheus — o padrão de fato."""
    ordenadas = sorted(LATENCIAS)
    p95 = ordenadas[int(0.95 * len(ordenadas)) - 1] if ordenadas else 0
    linhas = [
        "# HELP atlas_requisicoes_total Requisições atendidas",
        "# TYPE atlas_requisicoes_total counter",
        f"atlas_requisicoes_total {CONTADORES['requisicoes']}",
        "# TYPE atlas_erros_total counter",
        f"atlas_erros_total {CONTADORES['erros']}",
        "# TYPE atlas_pedidos_total counter",
        f"atlas_pedidos_total {CONTADORES['pedidos']}",
        "# TYPE atlas_latencia_p95_ms gauge",
        f"atlas_latencia_p95_ms {p95:.1f}",
    ]
    from fastapi.responses import PlainTextResponse
    return PlainTextResponse("\n".join(linhas) + "\n")


CONTADORES.update({"requisicoes": 1847, "erros": 12, "pedidos": 63})
LATENCIAS.extend([50] * 95 + [4000] * 5)

with Servico(metricas, "metricas") as s:
    print(httpx.get(f"{s.url}/metricas").text)

print("🔴 E a métrica em memória tem o problema do 09_01: com 4 workers,")
print("   cada um conta o seu pedaço. Em produção, use um `pushgateway`,")
print("   um coletor por processo, ou o `prometheus_client` em modo")
print("   multiprocesso.")

## 8. 🔧 Checklist de produção

In [ ]:
CHECKLIST = BASE / "CHECKLIST_PRODUCAO.md"
CHECKLIST.write_text("""# Checklist de produção — Atlas

## 🔴 Segurança
- [ ] Nenhum segredo no código nem na imagem (M06, M08)
- [ ] HTTPS com renovação automática **testada** (09_01)
- [ ] `forwarded_allow_ips` restrito ao proxy (09_01)
- [ ] SSH só por chave; root sem login (09_02)
- [ ] Processo roda como usuário sem privilégio (M08)
- [ ] Cabeçalhos de segurança no proxy (M06)
- [ ] Autorização testada por papel (M07)

## 🔴 Dados
- [ ] Backup automático
- [ ] **A restauração foi testada** — backup não testado não é backup
- [ ] Migrações compatíveis e rodadas antes do código (09_02)
- [ ] Volume nos serviços com estado (M08)

## Aplicação
- [ ] Configuração vem do ambiente (M06)
- [ ] Log estruturado no stdout (M04, 09_01)
- [ ] Sem estado no processo (09_01)
- [ ] Encerra com educação no SIGTERM (09_01)
- [ ] `/saude` e `/pronto` separados (M07)
- [ ] Timeout em todo cliente HTTP (M07)

## Servidor
- [ ] 🔴 `--reload` DESLIGADO
- [ ] Workers declarados, não calculados (09_01)
- [ ] `systemd` ou orquestrador reinicia se cair (09_02)
- [ ] Limites de CPU e memória (M08)

## Deploy
- [ ] Pipeline roda testes e auditoria (09_03)
- [ ] Rollback em um comando (09_02)
- [ ] Deploy verifica a saúde e reverte sozinho (09_02)
- [ ] Artefato construído uma vez e promovido (09_01)

## Monitoramento
- [ ] Os quatro sinais (latência p95, tráfego, erro, saturação)
- [ ] 🎯 Ao menos uma métrica de NEGÓCIO
- [ ] Alertas só para o que exige ação imediata
- [ ] Alerta de certificado a vencer
- [ ] Log sem dado pessoal (LGPD)

## Documentação
- [ ] `docs/DEPLOY.md` com o passo a passo
- [ ] Quem chamar quando quebrar
- [ ] O que fazer nos três incidentes mais prováveis
""", encoding="utf-8")

texto = CHECKLIST.read_text(encoding="utf-8")
itens = texto.count("- [ ]")
print(f"✅ {CHECKLIST.name} — {itens} itens\n")
print(texto)

> 🎯 **Olhe a coluna de origem: quase todo item veio de um módulo anterior.**
>
> Este checklist não é uma lista nova de trabalho — é o inventário do que você já construiu, organizado pela pergunta *"isto está pronto para produção?"*.
>
> 💭 **Os dois itens que ninguém faz, e que mais doem:**
>
> - **"A restauração foi testada"** — todo mundo tem backup; quase ninguém já restaurou um. Descobrir que o backup estava corrompido é uma experiência que se tem uma vez só.
> - **"O que fazer nos três incidentes mais prováveis"** — escrito com calma, hoje, para ser lido às 3h da manhã por alguém com sono.

## 📝 Exercícios

**E1.** Escreva um `ci.yml` com quatro jobs e dependências. Rode com o `rodar_workflow()`.

**E2.** Ordene as etapas do seu CI por custo crescente. Meça o tempo de cada uma.

**E3.** 🔴 Adicione um job que falhe se houver segredo no código. Commite um segredo e veja barrar.

**E4.** Adicione um job que rode a auditoria do M08. Provoque a falha.

**E5.** Estenda o `rodar_workflow()` para suportar `if:` simples (`success()`, `failure()`).

**E6.** 🔴 Demonstre o vazamento de segredo por transformação: `base64`, substring, `rev`. Explique por que o mascaramento não pega.

**E7.** Explique o que fazer quando um segredo vaza no log. Por que apagar o log não basta?

**E8.** Escreva um `cd.yml` com `environment` de aprovação manual e rollback com `if: failure()`.

**E9.** Argumente, por escrito, se a Aurora deve ter deploy automático em produção hoje. Use critérios, não opinião.

**E10.** Implemente o `FormatadorJSON` e adicione id de correlação em toda linha (M06).

**E11.** 🔴 Escreva um filtro de campos sensíveis. Depois converta a lista de bloqueio em lista de permissão e explique por que é mais segura.

**E12.** Calcule média, mediana, p95 e p99 de uma amostra com cauda longa. Explique por que a média mente.

**E13.** Escolha três métricas de negócio para a Aurora e justifique cada uma.

**E14.** 🔴 Classifique dez condições em "acorda", "horário comercial" e "só gráfico". Justifique as que não acordam ninguém.

**E15.** Implemente `/metricas` no formato Prometheus. Explique o problema dele com vários workers.

**E16.** Percorra o checklist de produção com o SEU Atlas e liste as pendências.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

In [ ]:
# E16

## 📋 Cola de referência

```yaml
# ═══ .github/workflows/ci.yml ═══
name: CI
on:
  push: {branches: [main]}
  pull_request:
env: {ATLAS_AMBIENTE: teste}

jobs:
  qualidade:                    # 🔑 ordem por CUSTO CRESCENTE
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: ruff check src/
  testes:
    needs: [qualidade]          # 🔑 o grafo de dependências
    steps:
      - run: pytest -q -m "not integracao"
  publicar:
    needs: [testes]
    if: github.ref == 'refs/heads/main'
    environment: producao       # 🔒 aprovação manual
    steps:
      - run: ./deploy.sh producao
        env: {CHAVE: "${{ secrets.CHAVE_DEPLOY }}"}
      - run: ssh ... rollback.sh
        if: failure()           # 🔴 reverte sozinho
```

```bash
# ═══ 🔴 Segredos no CI ═══
# mascaramento = substituição de texto EXATO
echo $CHAVE            ✅ ***
echo $CHAVE | base64   🔴 VAZA      echo ${CHAVE:0:8}  🔴 VAZA
set -x                 🔴 VAZA      curl -v            🔴 VAZA
# vazou? ROTACIONE. Apagar o log não basta.
# 🔒 melhor: OIDC — credencial temporária, sem segredo guardado

# ═══ Logs ═══
# JSON no stdout, com req_id em toda linha
# 🔴 nunca: senha, token, cartão, CPF, e-mail (LGPD)
#    registre usuario_id, não o e-mail

# ═══ Métricas — 4 sinais + 1 ═══
# latência (p95/p99, 🔴 NUNCA média) · tráfego · erros · saturação
# 🎯 + métrica de NEGÓCIO: pedidos/hora

# ═══ Alertas ═══
# só acorda o que exige ação IMEDIATA de uma pessoa
# 🔴 alerta que não exige ação deve ser APAGADO, não ignorado
```

## ✅ Checklist de saída

**Pipeline**

- [ ] Distingo CI de CD, e Delivery de Deployment
- [ ] Ordeno as etapas por custo crescente
- [ ] Uso `needs:` para o grafo de dependências
- [ ] Sei rodar o pipeline localmente antes do push
- [ ] Tenho um portão que barra segredo no código

**Segredos**

- [ ] 🔴 **Sei que o mascaramento é comparação de texto exato**
- [ ] Nunca imprimo segredo, nem parcialmente
- [ ] Sem `set -x` e sem `curl -v` em job com segredo
- [ ] Sei que vazou = rotacionar
- [ ] Conheço OIDC como alternativa

**Deploy automático**

- [ ] Sei os critérios para automatizar
- [ ] Uso `environment` para aprovação manual
- [ ] O deploy verifica a saúde
- [ ] 🔴 **Reverte sozinho com `if: failure()`**

**Observabilidade**

- [ ] Log em JSON, no stdout
- [ ] Id de correlação em toda linha
- [ ] 🔴 **Nenhum dado pessoal no log**
- [ ] Meço p95, não média
- [ ] 🎯 **Tenho ao menos uma métrica de negócio**
- [ ] Meus alertas exigem ação imediata
- [ ] Sei que métrica em memória quebra com N workers

**Produção**

- [ ] Percorri o checklist com o meu projeto
- [ ] 🔴 **Testei a RESTAURAÇÃO do backup**
- [ ] Escrevi o que fazer nos três incidentes mais prováveis

---

### ➡️ Próxima aula

**`09_99_Lista_Exercicios.ipynb`** — A lista do módulo e o projeto: o pipeline completo do Atlas, do `git push` ao ar.